In [ ]:
from pathlib import Path
import csv
import json
from itertools import islice


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "cross-data").exists() and (PROJECT_ROOT.parent / "cross-data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "cross-data"
OUTPUT_DIR = PROJECT_ROOT / "batch_inputs"

DOMAINS = ["news", "social-fb", "transcripts"]
SPLITS = ["test"]
#Change to ["train", "val", "test"] depending on which splits you want to process

ID_FIELD = "id"
TEXT_FIELD = "text"

# Change the settings below depending on the prompt strategy used. The settings below are for the "CoT" prompt strategy. For the "zero-shot" and "few-shot" prompt strategy, set REASONING_EFFORT and VERBOSITY to None, TEMPERATURE = 0 and MAX_COMPLETION_TOKENS = 256.

MODEL = "gpt-5-mini-2025-08-07"
#MAX_COMPLETION_TOKENS = 256
MAX_COMPLETION_TOKENS = 512
#REASONING_EFFORT = None
REASONING_EFFORT = "low"
#VERBOSITY = None
VERBOSITY = "low"
#TEMPERATURE = 0
TEMPERATURE = None
SEED = 42
RESPONSE_FORMAT = "none"

MAX_REQUESTS_PER_FILE = 40_000
ROW_LIMIT = None  

BASENAME = "batch_inputs"
MANIFEST_PATH = OUTPUT_DIR / "manifest.jsonl"


In [ ]:
SYSTEM_PROMPT = (
    "You are an expert analyst of propaganda and persuasive language.\n"
    "Your task is to identify whether a given text uses any of the persuasion techniques listed below.\n"
)

#This is the CoT prompt, copy paste the other ones from the folder "Prompt" if you want to use them instead.

USER_TEMPLATE = (
"""
Identify which of the following propaganda techniques are used in the text

- Loaded Language: Using specific words and phrases with strong emotional implications (either positive or negative) to influence and convince the audience that an argument is valid/true. This fallacy is also known as euphemisms, appeal to/argument from emotive language, or loaded language.

- Name Calling: A form of argument in which loaded labels are directed at an individual or group, typically in an insulting or demeaning way. Labelling an object as either something the target audience fears, hates, or on the contrary finds desirable or loves. This technique calls for a qualitative judgement that disregards facts and focuses solely on the essence of the subject being characterized. This technique is in a way also a manipulative wording, as it is used a the level of the nominal group rather than being a full-fledged argument with premise and conclusion. For example, in the political discourse, typically one is using adjectives and nouns as labels that refer to political orientation, opinions, personal characteristic, and association to some organisations, as well a insults. What distinguishes it from the Loaded Language technique, is that it is only concerned about the characterization of the subject.

- Appeal to Fear: This technique aims at promoting or rejecting through the repulsion of the audience towards this idea (e.g., via exploiting some preconceived judgements) or towards its alternative. The alternative could be the status quo, in which case the current situation is described in a scary way with Loaded Language. If the fear is linked with the consequences of a decision, it is often the case that this technique is used simultaneously with Appeal to Consequences (see Simplification techniques), and if the there are only two alternatives which are stated explicitly, then it is used simultaneously with False Dilemma technique.

- Flag Waving: Justifying or promoting an idea by exhaling the pride of a group or highlighting the benefits that specific group. The stereotypical example would be national pride, hence the name of the technique, however, the target group it applies to might be any group, e.g., related to race, gender, political preference, etc. The connection to nationalism , patriotism or benefit for an idea, group or country might be fully undue and is usually based on the presumption that the recipients already have certain beliefs, biases, and prejudices about the given issue. It can be seen as an appeal to emotions instead to logic of the audience aiming to manipulate them to win an argument. As such, this technique can also appear outside the form of well constructed argument, by simply making mentions that make resonate with the feeling of a particular group and as such setting up a context for further arguments.

- Bandwagon: This technique gives weight to an argument or idea by justifying it on the basis that allegedly `everybody' (or the large majority) agrees with it or `nobody' disagrees with it. As such the target audience is encouraged to gregariously adopt the same idea by
considering ``everyone else'' as an authority, and to join in and take the course of the same action. Here, ``everyone else'' might refer to the general public, key entities and actors in a certain domain, countries, etc. Analogously, an attempt to persuade the audience not to do something because ``nobody else is taking the same action'' falls under our definition of Bandwagon.

- Appeal to Authority: This technique gives weight to a claim or thesis by simply stating that a particular entity considered as an authority,
e.g., a person or an organisation, is the source of the information. The entity mentioned as an authority may, but does not need to be an actual valid authority in the domain-specific field to discuss a particular topic or to be considered and serve as an expert. What is important, and makes it different than simply sourcing an information, is that the tone of the text indicates that it capitalises on the weight of an alleged authority in order to justify some information, claim, or conclusion. Reference to a valid authority is not a logical fallacy, a reference to an invalid authority is, and both are captured within this label. In particular, a self-reference as an authority falls under Appeal to Authority too. When the arguer mentions the name of an authority or a group of people who agreed with her claim either without providing any relevant evidence, or by mentioning popular non-experts, or the acceptance of the claim by the majority.

- Slogans: A brief and striking phrase that may include labeling and stereotyping. Slogans tend to act as emotional appeals.

Read the text below and for each technique listed above:
First, check if the text uses it and return yes or no, along with the detected span(s) (verbatim) if yes.
Only return yes if you are extremely confident (confidence >= 0.99).

At the end, wrap your final answer in curly braces listing ONLY the names of the technique you answered yes to, separated by commas e.g. {Slogans, Bandwagon} — or {None} if none apply.


Text:
\"\"\"
{INPUT_TEXT}
\"\"\"
"""
)


In [ ]:
def iter_rows(path, max_samples):
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        if not reader.fieldnames:
            raise SystemExit(f"Empty TSV: {path}")
        missing = [field for field in (ID_FIELD, TEXT_FIELD) if field not in reader.fieldnames]
        if missing:
            raise SystemExit(f"Missing columns in {path}: {', '.join(missing)}")
        for idx, row in enumerate(reader):
            if max_samples is not None and idx >= max_samples:
                break
            yield row


def render_template(template, context):
    rendered = template
    for key, value in context.items():
        rendered = rendered.replace(f"{{{key}}}", value)
    return rendered


def build_messages(system_prompt, user_template, context):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": render_template(system_prompt, context)})
    messages.append({"role": "user", "content": render_template(user_template, context)})
    return messages


def sanitize_custom_id(value):
    return value.replace("|", "_").replace("\n", " ").strip()


def part_path(part_idx):
    return OUTPUT_DIR / f"{BASENAME}_part{part_idx:04d}.jsonl"


In [ ]:
rows_per_domain = {}
sample_rows = []

for domain in DOMAINS:
    for split in SPLITS:
        dataset_path = DATA_DIR / domain / f"{split}.tsv"
        if not dataset_path.exists():
            raise SystemExit(f"Missing dataset file: {dataset_path} (cwd={Path.cwd()})")
        count = 0
        for row in iter_rows(dataset_path, max_samples=None):
            count += 1
            if len(sample_rows) < 5:
                sample_rows.append((domain, split, row.get(ID_FIELD), row.get(TEXT_FIELD, "")[:140]))
        rows_per_domain[(domain, split)] = count

rows_per_domain

sample_rows


In [ ]:
def write_batches():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    part_idx = 1
    in_part = 0
    total_written = 0

    out_f = part_path(part_idx).open("w", encoding="utf-8")
    manifest_f = None
    if MANIFEST_PATH:
        manifest_f = MANIFEST_PATH.open("w", encoding="utf-8")

    try:
        for domain in DOMAINS:
            for split in SPLITS:
                dataset_path = DATA_DIR / domain / f"{split}.tsv"
                for row in iter_rows(dataset_path, ROW_LIMIT):
                    row_id = str(row.get(ID_FIELD, "")).strip()
                    text = str(row.get(TEXT_FIELD, "")).strip()
                    if not row_id or not text:
                        continue

                    custom_id = f"{domain}|{split}|{sanitize_custom_id(row_id)}"
                    context = {
                        "DOMAIN": domain,
                        "SPLIT": split,
                        "ID": row_id,
                        "INPUT_TEXT": text,
                    }
                    messages = build_messages(SYSTEM_PROMPT, USER_TEMPLATE, context)

                    body = {
                        "model": MODEL,
                        "messages": messages
                    }
                    if TEMPERATURE is not None:
                        body["temperature"] = TEMPERATURE
                    if SEED is not None:
                        body["seed"] = SEED
                    if MAX_COMPLETION_TOKENS is not None:
                        body["max_completion_tokens"] = MAX_COMPLETION_TOKENS
                    if RESPONSE_FORMAT == "json_object":
                        body["response_format"] = {"type": "json_object"}
                    if REASONING_EFFORT is not None:
                        body["reasoning_effort"] = REASONING_EFFORT
                    if VERBOSITY is not None:
                        body["verbosity"] = VERBOSITY

                    request = {
                        "custom_id": custom_id,
                        "method": "POST",
                        "url": "/v1/chat/completions",
                        "body": body,
                    }

                    out_f.write(json.dumps(request, ensure_ascii=True) + "\n")
                    if manifest_f:
                        manifest_record = {
                            "custom_id": custom_id,
                            "domain": domain,
                            "split": split,
                            "id": row_id,
                            "text": text,
                        }
                        manifest_f.write(json.dumps(manifest_record, ensure_ascii=True) + "\n")

                    in_part += 1
                    total_written += 1

                    if in_part >= MAX_REQUESTS_PER_FILE:
                        out_f.close()
                        part_idx += 1
                        in_part = 0
                        out_f = part_path(part_idx).open("w", encoding="utf-8")

    finally:
        out_f.close()
        if manifest_f:
            manifest_f.close()

    last_path = part_path(part_idx)
    if last_path.exists() and last_path.stat().st_size == 0:
        last_path.unlink()

    files = sorted(OUTPUT_DIR.glob(f"{BASENAME}_part*.jsonl"))
    print(f"Done. Total requests written: {total_written}")
    print(f"Number of batch input files: {len(files)}")
    for path in files[:5]:
        print(f"  {path}")
    if len(files) > 5:
        print("  ...")


write_batches()


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "" ### Add your OpenAI API key here

from openai import OpenAI

client = OpenAI()

api_key = os.environ.get("OPENAI_API_KEY")

batch_input_file = client.files.create(
    file=open("batch_inputs_part0001.jsonl", "rb"), #adjust the file path
    purpose="batch",
)

print(batch_input_file.id)


In [ ]:
from openai import OpenAI
client = OpenAI()

#Copy-paste the file ID from the previous cell output in "input_file_id"

batch = client.batches.create(
    input_file_id="", #Copy-paste the file ID from the previous cell output in "input_file_id"
    endpoint="/v1/chat/completions",
    completion_window="24h",
)

print(batch.id, batch.status)
